In [3]:
# -------------------- Authentication --------------------
import ee
ee.Authenticate()
ee.Initialize(project='ace-axon-461214-e5')

In [11]:

import geemap
import datetime
from IPython.display import display
import numpy as np
from sklearn.metrics import confusion_matrix  # For optional comparative analysis
import matplotlib.pyplot as plt  # For visualization and saving masks

# -------------------- Defaults (Updated for 2025 Flood Event over Punjab) --------------------
DEFAULT_PRE_START = '2025-06-01'
DEFAULT_PRE_END = '2025-07-30'
DEFAULT_POST_START = '2025-08-13'
DEFAULT_POST_END = '2025-09-30'

MIN_SLOPE_DEG = 10
DIFF_THRESHOLD_SAR = -3.0  # dB decrease for SAR flooding
NDWI_THRESHOLD = 0.3  # Threshold for water in NDWI (optical)

AREA_SCALE = 30
TILE_SCALE = 4
MOSAIC_METHOD = ee.Reducer.mean()


# -------------------- Load Geometry --------------------
punjab_geom = ee.Geometry.Polygon([
    [
        [74.85213182644908, 31.031667867569908],
        [75.18996141629283, 31.031667867569908],
        [75.18996141629283, 31.292547287669727],
        [74.85213182644908, 31.292547287669727],
        [74.85213182644908, 31.031667867569908]
    ]
])

# -------------------- SAR Helpers (from original) --------------------
def to_natural(img):
    return ee.Image(10.0).pow(img.select('VH').divide(10.0)).rename('VH')

def to_db(img):
    return ee.Image(img.select('VH')).log10().multiply(10.0).rename('VH')

def refined_lee(img):
    return img.focal_median(3) # Using a 3x3 median filter as a generic speckle reducer

def get_mosaic(col, reducer, geom):
    sorted_col = col.sort('system:time_start')
    return sorted_col.reduce(reducer).clip(geom).rename('VH')

# -------------------- Sentinel-2 --------------------
# Cloud masking function for Sentinel-2
def mask_s2_clouds(image):
    qa = image.select('QA60')
    # Bits 10 and 11 are clouds and cirrus, respectively.
    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11
    # Both flags should be set to zero, indicating clear conditions.
    mask = qa.bitwiseAnd(cloud_bit_mask).eq(0).And(qa.bitwiseAnd(cirrus_bit_mask).eq(0))
    # Return the masked image, scaled to reflectance (bands already TOA, but we use for NDWI)
    return image.updateMask(mask).divide(10000)  # Scale to 0-1 for visualization

# Compute NDWI (Normalized Difference Water Index) for water detection
def compute_ndwi(image):
    green = image.select('B3')
    nir = image.select('B8')
    ndwi = nir.subtract(green).divide(nir.add(green)).rename('NDWI')
    return image.addBands(ndwi)

# Get S2 mosaic with cloud masking and NDWI computation
def get_s2_mosaic(start_date, end_date, geom):
    s2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)) \
        .filterDate(start_date, end_date) \
        .filterBounds(geom) \
        .map(mask_s2_clouds) \
        .map(compute_ndwi)
    if s2.size().getInfo() == 0:
        raise ValueError(f"No S2 images found for {start_date} to {end_date}")
    mosaic = s2.median().clip(geom)  # Median composite for optical
    return mosaic

# -------------------- Change Detection Methods --------------------
# Basic Change Detection: For SAR (backscatter difference), for Optical (NDWI difference)
def sar_change_detection(pre_sar, post_sar):
    pre_nat = to_natural(pre_sar)
    post_nat = to_natural(post_sar)
    pre_filtered = to_db(refined_lee(pre_nat))
    post_filtered = to_db(refined_lee(post_nat))

    delta = post_filtered.subtract(pre_filtered).rename('delta')
    change_mask = delta.lt(DIFF_THRESHOLD_SAR).rename('change_sar').selfMask()

    # Auxiliary masking (slope and permanent water)
    elev = ee.Image("WWF/HydroSHEDS/03VFDEM").clip(punjab_geom).rename('elevation')
    slope = ee.Algorithms.Terrain(elev).select('slope').resample('bilinear').reproject(
        crs='EPSG:4326', scale=AREA_SCALE
    ).rename('slope')
    gsw = ee.Image("JRC/GSW1_2/GlobalSurfaceWater").select('seasonality').clip(punjab_geom)
    permanent_water = gsw.gte(5).rename('permanent')

    change_mask = change_mask.where(permanent_water, 0).selfMask()
    change_mask = change_mask.updateMask(slope.lt(MIN_SLOPE_DEG))

    # Confidence map: Use absolute delta value as confidence (higher change = higher confidence)
    confidence = delta.abs().rename('confidence_sar')

    return change_mask, confidence, pre_filtered, post_filtered

def optical_change_detection(pre_s2, post_s2):
    pre_ndwi = compute_ndwi(pre_s2).select('NDWI')
    post_ndwi = compute_ndwi(post_s2).select('NDWI')

    delta_ndwi = post_ndwi.subtract(pre_ndwi).rename('delta_ndwi')
    # Water change: Increase in NDWI > threshold indicates flooding
    change_mask = delta_ndwi.gt(NDWI_THRESHOLD).rename('change_optical').selfMask()

    # Confidence: Use delta NDWI value (higher increase = higher confidence)
    confidence = delta_ndwi.rename('confidence_optical')

    return change_mask, confidence, pre_ndwi, post_ndwi

# Fused Change Detection (Simple logical OR of SAR and Optical masks for union)
def fused_change_detection(sar_mask, optical_mask, sar_conf, optical_conf):

    fused_mask = sar_mask.Or(optical_mask).rename('change_fused')
    # Confidence: Average of normalized confidences (simple fusion)
    conf_sar_norm = sar_conf.divide(ee.Number(20))  # Normalize ~max dB change
    conf_opt_norm = optical_conf.divide(ee.Number(1))  # NDWI max ~1
    fused_conf = conf_sar_norm.addBands(conf_opt_norm).reduce(ee.Reducer.mean()).rename('confidence_fused')
    return fused_mask, fused_conf


# -------------------- Export/ "Download" Function --------------------
# Export change masks and confidences to Google Drive (programmatic "download")
def export_change_maps(change_mask, confidence, prefix):
    task1 = ee.batch.Export.image.toDrive(
        image=change_mask,
        description=f'{prefix}_mask',
        folder='Flood_Change_Detection',
        region=punjab_geom,
        scale=AREA_SCALE,
        maxPixels=1e13
    )
    task1.start()

    task2 = ee.batch.Export.image.toDrive(
        image=confidence,
        description=f'{prefix}_confidence',
        folder='Flood_Change_Detection',
        region=punjab_geom,
        scale=AREA_SCALE,
        maxPixels=1e13
    )
    task2.start()


# -------------------- Main Processing --------------------
def process_change_detection():
    # SAR Processing (Original + Accumulated for Post)
    s1 = ee.ImageCollection('COPERNICUS/S1_GRD') \
        .filter(ee.Filter.eq('instrumentMode', 'IW')) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH')) \
        .filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING')) \
        .filterBounds(punjab_geom) \
        .select('VH')

    pre_col_sar = s1.filterDate(DEFAULT_PRE_START, DEFAULT_PRE_END)
    post_col_sar = s1.filterDate(DEFAULT_POST_START, DEFAULT_POST_END)

    pre_count_sar = pre_col_sar.size().getInfo()
    post_count_sar = post_col_sar.size().getInfo()


    if pre_count_sar == 0 or post_count_sar == 0:
        raise ValueError('No SAR images found.')

    pre_sar = get_mosaic(pre_col_sar, MOSAIC_METHOD, punjab_geom)
    post_sar = get_mosaic(post_col_sar, MOSAIC_METHOD, punjab_geom)

    # For accumulated SAR flooding
    def process_single_post_sar(p_img):
        p_nat = to_natural(p_img)
        p_filtered = to_db(refined_lee(p_nat))
        delta = p_filtered.subtract(to_db(refined_lee(to_natural(pre_sar)))).rename('delta')
        flooded_single = delta.lt(DIFF_THRESHOLD_SAR).rename('water').selfMask()
        elev = ee.Image("WWF/HydroSHEDS/03VFDEM").clip(punjab_geom).rename('elevation')
        slope = ee.Algorithms.Terrain(elev).select('slope').resample('bilinear').reproject(
            crs='EPSG:4326', scale=AREA_SCALE
        ).rename('slope')
        gsw = ee.Image("JRC/GSW1_2/GlobalSurfaceWater").select('seasonality').clip(punjab_geom)
        permanent_water = gsw.gte(5).rename('permanent')
        flooded_single = flooded_single.where(permanent_water, 0).selfMask()
        flooded_single = flooded_single.updateMask(slope.lt(MIN_SLOPE_DEG))
        return flooded_single

    post_individual_sar = post_col_sar.map(process_single_post_sar)
    sar_flood_accum = post_individual_sar.max().rename('sar_flood_accum')

    # SAR Change Detection
    sar_mask, sar_conf, pre_sar_vis, post_sar_vis = sar_change_detection(pre_sar, post_sar)

    # Optical Processing
    pre_s2 = get_s2_mosaic(DEFAULT_PRE_START, DEFAULT_PRE_END, punjab_geom)
    post_s2 = get_s2_mosaic(DEFAULT_POST_START, DEFAULT_POST_END, punjab_geom)


    # Optical Change Detection
    optical_mask, optical_conf, pre_ndwi, post_ndwi = optical_change_detection(pre_s2, post_s2)

    # Fused
    fused_mask, fused_conf = fused_change_detection(sar_mask, optical_mask, sar_conf, optical_conf) # Pass confidence images

    # Visualization
    Map = geemap.Map(center=[30.835625, 75.411009], zoom=10)
    Map.addLayer(pre_sar_vis, {'min': -25, 'max': 0}, 'SAR Pre', True)
    Map.addLayer(post_sar_vis, {'min': -25, 'max': 0}, 'SAR Post', True)
    Map.addLayer(sar_mask, {'palette': ['red']}, 'SAR Change Mask', True)
    Map.addLayer(pre_ndwi, {'min': -1, 'max': 1, 'palette': ['blue', 'white', 'green']}, 'NDWI Pre', True)
    Map.addLayer(post_ndwi, {'min': -1, 'max': 1, 'palette': ['blue', 'white', 'green']}, 'NDWI Post', True)
    Map.addLayer(optical_mask, {'palette': ['blue']}, 'Optical Change Mask', True)
    #Map.addLayer(fused_mask, {'palette': ['purple']}, 'Fused Change Mask', True)
    display(Map)

    # Area Stats (example for SAR)
    pixel_area = ee.Image.pixelArea()
    inund_area_img = sar_mask.multiply(pixel_area)
    stats = inund_area_img.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=punjab_geom,
        scale=AREA_SCALE,
        maxPixels=1e13,
        tileScale=TILE_SCALE,
        bestEffort=True
    )
    inund_km2 = ee.Number(stats.get('change_sar', 0)).divide(1e6).getInfo()
    total_km2 = punjab_geom.area().divide(1e6).getInfo()
    perc = (inund_km2 / total_km2) * 100
    print(f'SAR Inundated: {inund_km2:.2f} km² ({perc:.2f}%) of {total_km2:.2f} km²')

    # -------------------- Visual Inspection and Annotation --------------------
    # For manual annotation: Sample points or regions (example: export sample for local annotation)
    sample_roi = ee.Geometry.Rectangle([74.5, 30.5, 75.5, 31.5])  # Example sub-ROI
    sample_sar_mask = sar_mask.clip(sample_roi)
    sample_opt_mask = optical_mask.clip(sample_roi)

    # Export sample for local Python annotation (using matplotlib for simple viz/save)
    geemap.ee_export_image(sample_sar_mask, filename='sample_sar_mask.tif', scale=30, region=sample_roi)
    geemap.ee_export_image(sample_opt_mask, filename='sample_opt_mask.tif', scale=30, region=sample_roi)
    #print('Sample masks exported for visual inspection/annotation.')

    # Generate random sample points over Punjab
    points = ee.FeatureCollection.randomPoints(punjab_geom, 1000, seed=42)

    # Create an image from the sar_mask and optical_mask to sample both simultaneously
    combined_for_sampling = sar_mask.rename('sar_change').addBands(optical_mask.rename('optical_change'))

    # Sample the combined image at the generated points
    sampled_features = combined_for_sampling.sampleRegions(
        collection=points,
        scale=AREA_SCALE,
        tileScale=TILE_SCALE
    ).getInfo()['features']

    sar_vals_list = []
    opt_vals_list = []

    # Extract values only for points where both SAR and Optical have valid data
    for f in sampled_features:
        props = f['properties']
        if 'sar_change' in props and props['sar_change'] is not None and \
           'optical_change' in props and props['optical_change'] is not None:
            sar_vals_list.append(props['sar_change'])
            opt_vals_list.append(props['optical_change'])

    sar_vals = np.array(sar_vals_list)
    opt_vals = np.array(opt_vals_list)

    # Confusion matrix (binary: 0=no change, 1=change)
    cm = confusion_matrix(sar_vals.astype(int), opt_vals.astype(int))

# Run
process_change_detection()

Map(center=[30.835625, 75.411009], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=Sear…

SAR Inundated: 51.81 km² (5.56%) of 932.47 km²
Generating URL ...
Please wait ...
Data downloaded to /content/sample_sar_mask.tif
Generating URL ...
Please wait ...
An error occurred while downloading.
